# Phase 2: Grounded Synthesis, Vector RAG & Dynamic Extraction Matrices
**Nexus Scholar Interactive Research Suite**

This notebook demonstrates how to query the indexed ChromaDB vector store with sectional slicing, extract dynamic protocol matrix dimensions across studies, and generate claim-entailed grounded synthesis reviews.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown
from scholar_protocol.models import ResearchProtocol
from scholar_rag.retriever import ScholarRetriever
from scholar_rag.synthesis import GroundedSynthesisEngine
from scholar_rag.matrix import MatrixExtractor

## 1. Section-Targeted Semantic Retrieval

In [ ]:
retriever = ScholarRetriever(db_path="./chroma_db")
results = retriever.query(
    query_text="computational complexity throughput scaling",
    section_category="results_empirical",
    n_results=5
)
print(f"Retrieved {len(results)} chunks from empirical results sections.")
for r in results[:2]:
    print(f"[{r.citation_token}] (Similarity: {r.cosine_sim:.3f})")
    print(f"{r.text[:200]}...\n")

## 2. Dynamic Protocol Matrix Extraction

In [ ]:
proto_path = Path("protocol.json")
if proto_path.exists():
    protocol = ResearchProtocol.model_validate(json.loads(proto_path.read_text(encoding="utf-8")))
    extractor = MatrixExtractor(protocol=protocol, retriever=retriever)
    rows, csv_path, json_path = extractor.extract_all(output_dir="./literature")
    print(f"Extracted matrix for {len(rows)} studies. Saved to {csv_path}")
    
    # Display Interactive Matrix Grid
    df_matrix = pd.DataFrame(rows)
    display(df_matrix.head())
else:
    print("Protocol file not found.")

## 3. Grounded Synthesis & Entailment Verification

In [ ]:
engine = GroundedSynthesisEngine(retriever=retriever)
synthesis_res = engine.synthesize(
    query="What throughput improvements are achieved?",
    rq_id="RQ1",
    section_category="results_empirical"
)

print(f"Synthesis Entailment Rate: {synthesis_res.entailment_rate * 100:.1f}%")
display(Markdown(synthesis_res.synthesis_markdown))